In [ ]:
# LiitLLM — ablation training, part 1 of 3
#
# Trains up to step 59999, pushes a checkpoint, and stops. Run part
# 2 next — on either account — and it resumes from here.
#
# Parts are cut by STEP, not by wall clock, so part boundaries are the
# same regardless of how fast the session's GPU happened to be.
#
# Accelerator must be T4 x2. Every cell below is safe to re-run.

In [ ]:
PART = 1
CONFIG = 'configs/ablation-unfiltered.yaml'
PREV_SLUG = '__none__'   # part 1 has no predecessor
STOP_AT_STEP = 59999
MAX_HOURS = 11.0   # backstop if the step estimate is off

In [ ]:
import torch, sys
assert torch.cuda.is_available(), "no GPU — set the accelerator to T4 x2"
cap = torch.cuda.get_device_capability()
name = torch.cuda.get_device_name(0)
print(f"{name}  sm_{cap[0]}{cap[1]}")
assert cap >= (7, 0), (
    f"{name} is sm_{cap[0]}{cap[1]}; Kaggle's PyTorch needs sm_70+. "
    "Set --accelerator NvidiaTeslaT4 (the default P100 will not work)."
)

In [ ]:
import glob
from pathlib import Path

def find_one(slug, filename, kind):
    """Locate a mounted source by slug. `kind` is 'datasets' or 'notebooks'.

    Kaggle mounts sources at /kaggle/input/<kind>/<owner>/<slug>/..., NOT at the
    flat /kaggle/input/<slug>/ that most examples show — a glob written for the
    flat layout silently matches nothing.

    `kind` is not optional, and that is the point. Every kernel output carries a
    full copy of the repo, so a slug-anchored search across all of /kaggle/input
    finds BOTH the repo dataset and the repo copy embedded in the previous part's
    output. Scoping the search to where the thing legitimately lives —
    the repo always in 'datasets', corpora and checkpoints always in 'notebooks'
    — makes the ambiguity impossible instead of merely detected.
    """
    hits = glob.glob(f"/kaggle/input/{kind}/**/{slug}/**/{filename}", recursive=True)
    assert len(hits) == 1, (
        f"expected exactly 1 {filename} under {kind}/{slug}, found {hits}.\n"
        f"Sources actually mounted: {sorted(glob.glob('/kaggle/input/*/*/*'))}"
    )
    return Path(hits[0])

In [ ]:
import shutil, os, sys

REPO_SLUG = "liitllm-repo"   # dataset holding this repo
PREP_SLUG = "00-prep"        # kernel whose OUTPUT holds the corpora + tokenizer

repo_src = find_one(REPO_SLUG, "pyproject.toml", "datasets").parent
REPO = Path("/kaggle/working/liitllm-repo")
if REPO.exists():
    shutil.rmtree(REPO)
shutil.copytree(repo_src, REPO)
sys.path.insert(0, str(REPO))
os.chdir(REPO)
print(f"repo: {repo_src} -> {REPO}")

In [ ]:
# The correctness gates. Both are free (CPU, ~1 min) and they are the entire
# reason any bad Taglish output can be blamed on the data instead of the code.
# Never skip them to save a minute at the start of a 20-hour run.
!python test_liitllm.py
!python test_resume.py

In [ ]:
# Point the config at the corpora in the prep kernel's output.
import yaml
cfg_path = REPO / CONFIG
cfg = yaml.safe_load(cfg_path.read_text())
arm = Path(cfg['data_dir']).name          # 'filtered' or 'unfiltered'
data_root = find_one(PREP_SLUG, 'tokenizer.json', 'notebooks').parent
cfg['data_dir'] = str(data_root / arm)
cfg['tokenizer_path'] = str(data_root / 'tokenizer.json')
cfg['out_dir'] = '/kaggle/working/out'
cfg_path.write_text(yaml.safe_dump(cfg))
print(yaml.safe_dump(cfg))

In [ ]:
# Restore the previous part's checkpoint from ITS kernel output. Loud
# either way: a silent fresh start is the worst failure here, because it
# looks exactly like progress for the next four hours.
import shutil, torch, glob
out = Path('/kaggle/working/out'); out.mkdir(parents=True, exist_ok=True)
prev = glob.glob(f'/kaggle/input/notebooks/**/{PREV_SLUG}/**/ckpt.pt', recursive=True)
RESUME = bool(prev)
if RESUME:
    src = Path(prev[0]).parent
    for f in src.iterdir():
        if f.is_file():
            shutil.copy(f, out)
    step = torch.load(out / 'ckpt.pt', weights_only=False)['step']
    print(f'=== RESUMING from {src} @ step {step} ===')
    assert step < STOP_AT_STEP, (
        f'checkpoint is already at step {step}, past this part\'s target '
        f'{STOP_AT_STEP} — you are running an earlier part than you meant to'
    )
else:
    print('=== NO CHECKPOINT - starting from step 0 (correct for part 1) ===')

In [ ]:
# Train this part.
#
# The try/except is the fallback. Whatever happens, /kaggle/working/out
# already holds the last atomically-written checkpoint, and it becomes
# this kernel's output either way — so an OOM or a CUDA error at hour 3
# costs one eval interval, not the part. The repo copy is deleted first so
# the output carries only the checkpoint, keeping later globs unambiguous.
from liitllm.train import train
err = None
try:
    train(str(cfg_path), resume=RESUME, max_hours=MAX_HOURS,
          stop_at_step=STOP_AT_STEP)
except Exception as e:
    err = e
    print(f'TRAINING FAILED: {type(e).__name__}: {e}')
    print('The last checkpoint is still in /kaggle/working/out and will be '
          'saved as this kernel\'s output. Rerun this part to continue.')
# Drop the repo copy so this kernel's output is ONLY the checkpoint.
# Left in place it reappears in the next part's /kaggle/input, where it
# doubles the upload and makes a repo lookup ambiguous.
import shutil, os
os.chdir('/kaggle/working')
shutil.rmtree(REPO, ignore_errors=True)
print(sorted(p.name for p in out.iterdir()))

In [ ]:
# Where did this part land, and is the run still worth continuing?
#   still descending -> run the next part
#   flattened        -> stop extending; grow the corpus instead
#   turned upward    -> memorising; the best checkpoint is already behind us
import csv
from liitllm.train import plot_curve
plot_curve(out / 'loss.csv')
vals = [(int(r['step']), float(r['val_loss']))
        for r in csv.DictReader((out / 'loss.csv').open())]
best = min(vals, key=lambda v: v[1])
print(f'best val {best[1]:.4f} @ step {best[0]}   '
      f'last val {vals[-1][1]:.4f} @ step {vals[-1][0]}')
print('OVERFITTING - best checkpoint is behind us'
      if best[0] < vals[-1][0] else 'still improving - run the next part')